# Generative AI Evaluation & Benchmarking

Evaluating generative AI models requires moving beyond single accuracy numbers:
1. **Lexical & Reference-based Metrics**: Exact Match, Token F1, ROUGE-L, and BLEU.
2. **RAG & Groundedness Metrics**: Claim-level citation support, hallucination rate, and query relevance.
3. **Information Retrieval Metrics**: Recall@k, Mean Reciprocal Rank (MRR), and nDCG@k.
4. **LLM-as-a-Judge**: Rubric scoring, pairwise comparisons, and position bias mitigation via swap-trial analysis.
5. **Human Evaluation & Statistics**: Inter-annotator agreement (Cohen's Kappa) and non-parametric bootstrap confidence intervals.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('06-generative-ai/evaluation/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import eval_metrics
print('eval_metrics loaded')

## 1. Lexical Overlap Metrics: Exact Match, Token F1, ROUGE-L, BLEU
Deterministic baseline comparison between generated output and reference target.

In [ ]:
from eval_metrics import exact_match, token_f1, rouge_l, bleu_score

pred = "The transformer architecture relies on multi head self attention mechanisms."
ref = "The transformer architecture uses multi-head self-attention mechanisms for sequences."

print("Exact match:", exact_match(pred, ref))
print("Token F1   :", f"{token_f1(pred, ref):.4f}")
rl = rouge_l(pred, ref)
print(f"ROUGE-L    : P={rl['precision']:.3f}, R={rl['recall']:.3f}, F1={rl['f1']:.3f}")
print("Sentence BLEU:", f"{bleu_score(pred, ref):.4f}")

## 2. RAG Quality: Groundedness, Hallucination & Query Relevance
Verifying that statements made by the model are directly supported by context passages.

In [ ]:
from eval_metrics import evaluate_hallucination, query_relevance

context = (
    "The James Webb Space Telescope (JWST) was launched on December 25, 2021. "
    "It orbits around the Sun-Earth L2 Lagrange point approximately 1.5 million kilometers from Earth."
)

good_response = "JWST was launched on December 25, 2021. It orbits around the Sun-Earth L2 point."
hallucinated_response = "JWST was launched from Florida in July 2020. It orbits in low Earth orbit."

eval_good = evaluate_hallucination(good_response, context)
eval_bad = evaluate_hallucination(hallucinated_response, context)

print("Good Response Groundedness:", eval_good["groundedness"], "| Hallucination rate:", eval_good["hallucination_rate"])
print("Bad Response Groundedness :", eval_bad["groundedness"], "| Hallucination rate:", eval_bad["hallucination_rate"])

query = "When was the James Webb Space Telescope launched and where is it?"
print("Query Relevance (Good):", f"{query_relevance(query, good_response):.2f}")
print("Query Relevance (Irrelevant):", f"{query_relevance(query, 'The sun is a main-sequence star.'):.2f}")

## 3. Retrieval Metrics: Recall@k, MRR, nDCG@k
Evaluating the quality of the retriever subsystem in RAG pipelines.

In [ ]:
from eval_metrics import recall_at_k, mrr, ndcg_at_k

retrieved_docs = ["doc_10", "doc_4", "doc_1", "doc_9", "doc_3"]
ground_truth = {"doc_1", "doc_3"}

print(f"Recall@1 : {recall_at_k(retrieved_docs, ground_truth, k=1):.2f}")
print(f"Recall@3 : {recall_at_k(retrieved_docs, ground_truth, k=3):.2f}")
print(f"Recall@5 : {recall_at_k(retrieved_docs, ground_truth, k=5):.2f}")
print(f"MRR      : {mrr(retrieved_docs, ground_truth):.4f} (first hit at rank 3)")
print(f"nDCG@5   : {ndcg_at_k(retrieved_docs, ground_truth, k=5):.4f}")

## 4. LLM-as-a-Judge: Prompt Rubric & Position-Bias Mitigation
Using LLM judges safely requires structured criteria and neutralizing order biases.

In [ ]:
from eval_metrics import (
    format_rubric_judge_prompt,
    format_pairwise_judge_prompt,
    run_pairwise_swap_judge,
    parse_judge_score
)

prompt = format_rubric_judge_prompt(
    query="Explain backpropagation in one sentence.",
    response="Backpropagation calculates loss gradients with respect to weights using the chain rule.",
    rubric="Score 5 for mathematically sound concise definition; 1 for factually incorrect."
)
print("--- Judge Rubric Prompt ---")
print(prompt)

mock_judge_output = "Reasoning: Clear, concise, accurately references the chain rule.\nScore: 5"
print("Extracted score:", parse_judge_score(mock_judge_output))

## 5. Detecting & Neutralizing Position Bias in Pairwise Judging
If a judge always picks the first option (Option A), swapping orders detects the artifact and flags a Tie.

In [ ]:
# 1. Consistent evaluator
def fair_judge(q, a, b):
    return "Winner: A" if len(a) > len(b) else "Winner: B"

resp_long = "Gradient descent updates parameters iteratively in the negative gradient direction."
resp_short = "It updates weights using gradients."

fair_res = run_pairwise_swap_judge(fair_judge, "Explain GD", resp_long, resp_short)
print("Fair Judge Result:", fair_res)

# 2. Biased evaluator that always picks whichever is presented first
def biased_judge(q, a, b):
    return "Winner: A"

biased_res = run_pairwise_swap_judge(biased_judge, "Explain GD", resp_long, resp_short)
print("Biased Judge Result (Neutralized):", biased_res)

## 6. Human Evaluation Agreement & Bootstrap Confidence Intervals
Cohen's Kappa for inter-annotator agreement and statistical significance testing.

In [ ]:
from eval_metrics import cohens_kappa, bootstrap_ci

# Inter-rater agreement between two domain experts (1: Poor, 2: Acceptable, 3: Excellent)
expert_1 = [3, 2, 1, 3, 2, 3, 1, 2]
expert_2 = [3, 2, 1, 2, 2, 3, 1, 2]

kappa = cohens_kappa(expert_1, expert_2)
print(f"Cohen's Kappa agreement: {kappa:.3f} (Values > 0.8 indicate near-perfect agreement)")

# Bootstrap 95% Confidence Interval for benchmark scores
benchmark_scores = [0.85, 0.90, 0.78, 0.92, 0.88, 0.84, 0.89, 0.91, 0.86, 0.87]
low_ci, high_ci = bootstrap_ci(benchmark_scores, n_boot=1000, alpha=0.05)
mean_score = sum(benchmark_scores) / len(benchmark_scores)
print(f"Benchmark Mean: {mean_score:.3f} (95% CI: [{low_ci:.3f}, {high_ci:.3f}])")